<a href="https://colab.research.google.com/github/rockarus/csot-ml-astronomy/blob/main/submissions/arnesh/week2/week2_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import random
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision
import matplotlib.pyplot as plt

if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"
print("Device:", device)


Device: cuda


In [2]:
from pathlib import Path

RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2" / "images"
DATA_ROOT = Path("galaxy_data")
LABELS = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"

!pip -q install kaggle pandas
!kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
!unzip -q -o galaxy-zoo-2-images.zip -d {RAW_ROOT}
!wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz {LABELS}
!gunzip -c {RAW_ROOT}/gz2_hart16.csv.gz > {RAW_ROOT}/gz2_hart16.csv
print("RAW_ROOT   =", RAW_ROOT)
print("IMAGES_DIR =", IMAGES_DIR)
print("DATA_ROOT  =", DATA_ROOT)

import os
print(os.listdir(IMAGES_DIR))
jpg_files = list(Path(IMAGES_DIR).rglob("*.jpg"))
print("JPG count:", len(jpg_files))
head = pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3)
print(head)

Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.06G/3.06G [00:28<00:00, 116MB/s] 

RAW_ROOT   = galaxy_raw
IMAGES_DIR = galaxy_raw/images_gz2/images
DATA_ROOT  = galaxy_data
['159464.jpg', '9192.jpg', '116191.jpg', '46135.jpg', '106261.jpg', '256914.jpg', '183809.jpg', '237512.jpg', '88855.jpg', '195492.jpg', '218897.jpg', '64583.jpg', '113480.jpg', '269765.jpg', '168528.jpg', '183933.jpg', '29930.jpg', '110917.jpg', '61966.jpg', '196674.jpg', '127816.jpg', '77085.jpg', '89280.jpg', '3721.jpg', '281443.jpg', '110396.jpg', '275788.jpg', '255747.jpg', '196792.jpg', '218100.jpg', '33447.jpg', '137914.jpg', '171501.jpg', '130026.jpg', '99160.jpg', '252523.jpg', '89131.jpg', '242791.jpg', '277628.jpg', '52659.jpg', '116783.jpg', '151416.jpg', '68520.jpg', '227269.jpg', '47835.jpg', '64042.jpg', '55826.jpg', '180597.jpg', '237992.jpg', '122731.jpg', '200808.jpg', '59927.jpg', '144346.jpg', '16151.jpg', 

In [3]:
def high_level_label(gz2_class):
  if gz2_class.startswith("E"):
    return "elliptical"
  elif gz2_class.startswith("SB"):
    return "spiral_barred"
  elif gz2_class.startswith("S"):
    return "spiral"
  else:
    return None

def load_labeled_table(mapping_csv, labels_csv):
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    return df.dropna(subset=["label"]).reset_index(drop=True)

def _link_image(src, dst):
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(Path(src).resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(images_dir, df, out_root, per_class=200,
                                   train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    images_dir, out_root = Path(images_dir), Path(out_root)
    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)
        n = len(rows)
        n_train, n_val = int(train_frac * n), int(val_frac * n)
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train:n_train + n_val],
            "test": rows.iloc[n_train + n_val:],
        }
        for split_name, split_rows in splits.items():
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists():
                    _link_image(src, dst)


PER_CLASS = 20000

df = load_labeled_table(RAW_ROOT / "gz2_filename_mapping.csv", RAW_ROOT / "gz2_hart16.csv")
build_split_imagefolder_layout(IMAGES_DIR, df, DATA_ROOT, per_class=PER_CLASS)


In [4]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
num_classes = len(train_ds.classes)

In [5]:
import numpy as np
import torch.nn as nn
import torch.optim as optim

class GalaxyMLP(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.flatten = nn.Flatten()
    self.fc1=nn.Linear(3*64*64, 128)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(128, num_classes)
  def forward(self, x):
    x=self.flatten(x)
    x=self.fc1(x)
    x=self.relu(x)
    return self.fc2(x)

model = GalaxyMLP(num_classes=num_classes).to(device)

In [7]:
print(model)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total}, trainable parameters: {trainable}")


GalaxyMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)
Total parameters: 1573379, trainable parameters: 1573379


In [8]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

logits = model(images)
print(logits.device)
print(logits.shape)


cuda:0
torch.Size([32, 3])


In [11]:
import math
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss=criterion(logits, labels)
print(loss.item()-math.log(num_classes))

0.03603484229807674


In [25]:
model.train()
loss1=criterion(model(images), labels)

optimizer.zero_grad()
loss1.backward()
optimizer.step()

loss2=criterion(model(images), labels)
print(f"Loss decreased by {loss1-loss2}")

Loss decreased by 0.06535947322845459
